# Exercises (Student) - MCP Client with LLM

*Completed version — all TODOs filled in.*

In [ ]:
!pip install -q mcp nest_asyncio requests


In [ ]:
import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set

import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


Below we write out `server.py`. It exposes three tools (`add`, `multiply`, `greet`) and one resource (`info://server`).

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    "Multiply two numbers."
    return a * b

@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    return f"Hello, {name}!"

@mcp.resource("info://server")
def get_info() -> str:
    "Basic server info resource."
    return "DemoServer v1.0 - provides add, multiply, and greet tools."

if __name__ == "__main__":
    mcp.run()


## Exercise 1 (provide answer)

**Q: Why is STDIO transport simple for local MCP dev compared to HTTP?**

STDIO transport runs the MCP server as a plain child process and talks to it over its
standard input/output streams. That means:

- No networking setup at all: no port to pick, no binding to `localhost`, no firewall,
  CORS, or reverse-proxy configuration to worry about.
- No authentication/TLS layer is needed, since the pipe is private to the parent
  process that spawned the child — the OS itself scopes access, so it's inherently
  the most locked-down option for local dev.
- Lifecycle management is trivial: the client starts the server as a subprocess and
  the process exits (or is killed) when the client's context manager closes, so
  there's nothing left running in the background to clean up.
- It works identically on any machine (laptop, CI runner, Colab) with zero
  environment-specific configuration, whereas HTTP requires the server to already be
  reachable at some URL/port and often needs extra plumbing (uvicorn, ngrok, etc.)
  just to test locally.

HTTP transport becomes worth the extra setup once you need multiple simultaneous
clients, a server that lives independently of any one client, or remote access across
machines — none of which matter for local single-user development.

## Exercise 2

In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()


In [ ]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


## Exercise 3

In [ ]:
async def ex3_list():
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))


In [ ]:
await ex3_list()


## Exercise 4

**Q: Explain how the conversion to llm tool happens in MCP server code?**

When you decorate a function with `@mcp.tool()`, `FastMCP` inspects the function at
registration time rather than at call time. It reads:

- the function's **name**, which becomes the tool's `name`;
- its **docstring**, which becomes the tool's `description`;
- its **type-annotated parameters**, which FastMCP turns into a JSON-schema
  `inputSchema` — each argument's Python type (`int`, `str`, etc.) is mapped to a
  JSON-schema type (`integer`, `string`, ...) under `properties`, and any parameter
  without a default value is added to `required`.

That schema is what `session.list_tools()` returns for each tool (`t.inputSchema`).
`convert_to_llm_tool` then just re-shapes that MCP-native schema into the OpenAI-style
"function calling" spec that most LLM APIs expect: it copies `name` and
`description` straight across, and nests the `properties`/`required` fields from
`inputSchema` under `parameters` with `"type": "object"`. No new information is
invented — it's purely a reformatting step so the same schema MCP already computed
from the Python function signature can be handed to an LLM as a callable "tool".

In [ ]:
def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like "Add 2 to 20.

In [ ]:
import asyncio
import json
import re
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def stub_plan(prompt: str, functions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """A tiny rule-based stand-in for an LLM planner.

    Looks for simple natural-language patterns ('add X to Y', 'multiply X by Y',
    'greet NAME') and returns the matching MCP tool call(s), only if that tool is
    actually available on the connected server.
    """
    p = prompt.lower()
    available = {f["function"]["name"] for f in functions}

    def to_num(s):
        v = float(s)
        return int(v) if v.is_integer() else v

    m = re.search(r"add\s+(-?\d+(?:\.\d+)?)\s+to\s+(-?\d+(?:\.\d+)?)", p)
    if m and "add" in available:
        return [{"name": "add", "args": {"a": to_num(m.group(1)), "b": to_num(m.group(2))}}]

    m = re.search(r"multiply\s+(-?\d+(?:\.\d+)?)\s+by\s+(-?\d+(?:\.\d+)?)", p)
    if m and "multiply" in available:
        return [{"name": "multiply", "args": {"a": to_num(m.group(1)), "b": to_num(m.group(2))}}]

    m = re.search(r"greet\s+(\w+)", p)
    if m and "greet" in available:
        return [{"name": "greet", "args": {"name": m.group(1)}}]

    return []

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls

async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            functions = [convert_to_llm_tool(t) for t in tools.tools]
            calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])


In [ ]:
await ex5_run("Add 2 to 20")


## Optional - add multiply(a, b) and rerun

`multiply` was already added to `server.py` above and `stub_plan` already knows the
"multiply X by Y" pattern, so we just rebuild the function list (implicitly, inside
`ex5_run`) and rerun the planner/executor with a multiplication prompt.

In [ ]:
await ex5_run("Multiply 4 by 5")


**Extra sanity check** — greet tool, to show the planner/executor loop generalizes
beyond arithmetic:

In [ ]:
await ex5_run("Please greet Ada")
